# exp_20260513_024_lgb_progress_window_weather_agg_min

PS S6E5 Predicting F1 Pit Stops

Base: `020 LGB progress-window TE`

Add: Race-Year weather aggregate features from Woods Hole FastF1 weather dataset.

This notebook intentionally does not use `Normalized_TyreLife`, its proxy, weather-target TE, pseudo label, or time/RaceProgress interpolation.


In [1]:
# ============================================================
# PS S6E5 - exp_20260513_024_lgb_progress_window_weather_agg_min
#
# Base:
# - 020 LGB progress-window TE
#
# Add:
# - Race-Year weather aggregate features from:
#   woodshole/playground-series-s6-e5-weather-conditions
#
# Do NOT add:
# - Normalized_TyreLife
# - Normalized_TyreLife proxy / final stint endpoint proxy
# - Weather x target TE
# - Time x RaceProgress interpolation
# - Race_Year_RaceProgressBin
# - pseudo label
# ============================================================

In [2]:
import os
import re
import gc
import json
import time
import random
import warnings
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260513_024_lgb_progress_window_weather_agg_min"
    TAG = "lgb_progress_window_weather_agg_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    # Official original dataset. The exact file name can vary by Kaggle dataset version,
    # so we discover it by glob. If not found, this experiment still runs competition-only.
    ORIGINAL_ROOT_CANDIDATES = [
        Path("/kaggle/input/f1-strategy-dataset-pit-stop-prediction"),
        Path("/kaggle/input/f1-strategy-dataset-pit-stop-prediction/data"),
        Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction"),
    ]

    # Woods Hole FastF1 weather dataset.
    WEATHER_ROOT_CANDIDATES = [
        Path("/kaggle/input/playground-series-s6-e5-weather-conditions"),
        Path("/kaggle/input/datasets/woodshole/playground-series-s6-e5-weather-conditions"),
    ]
    WEATHER_FILENAME = "F1_Weather_2022_2025.csv"

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5
    INNER_SPLITS = 5

    USE_ORIGINAL_APPEND = True
    DROP_PITSTOP = True
    DROP_NORMALIZED_TYRELIFE = True

    RACE_PROGRESS_BINS = 20

    TE_COLS = [
        "RaceProgressBin_20",
        "Race_RaceProgressBin_20",
        "Race_Compound_RaceProgressBin_20",
    ]
    TE_SMOOTH = 20.0

    LGB_PARAMS = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.02,
        "n_estimators": 8000,
        "num_leaves": 64,
        "max_depth": -1,
        "min_child_samples": 80,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.15,
        "reg_lambda": 2.0,
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": -1,
    }


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def auc(y, p) -> float:
    return roc_auc_score(y, p)


def find_first_existing_file(roots, filename=None, pattern="*.csv"):
    for root in roots:
        if not root.exists():
            continue
        if filename is not None:
            p = root / filename
            if p.exists():
                return p
        files = list(root.glob(pattern)) + list(root.rglob(pattern))
        if files:
            return files[0]
    return None


def norm_text(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\b(grand|prix|gp|formula|1|f1|race|circuit|autodromo|autodrome|international|street|track)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def safe_div(a, b):
    return a / np.where(np.asarray(b) == 0, np.nan, b)


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("sub  :", sub.shape)
print("target mean:", train[CFG.TARGET].mean())

original_path = find_first_existing_file(
    CFG.ORIGINAL_ROOT_CANDIDATES,
    filename=None,
    pattern="*.csv",
)

orig = None
if original_path is not None:
    orig_tmp = pd.read_csv(original_path)
    if CFG.TARGET in orig_tmp.columns:
        orig = orig_tmp.copy()
        print("original:", original_path, orig.shape)
    else:
        print("original candidate found, but target column missing. skipped:", original_path)
else:
    print("original dataset not found. run competition-only.")

weather_path = find_first_existing_file(
    CFG.WEATHER_ROOT_CANDIDATES,
    filename=CFG.WEATHER_FILENAME,
    pattern="*.csv",
)
assert weather_path is not None, "Weather dataset not found. Add woodshole/playground-series-s6-e5-weather-conditions as Kaggle input."

weather = pd.read_csv(weather_path)
print("weather:", weather_path, weather.shape)
display(weather.head())

train: (439140, 16)
test : (188165, 15)
sub  : (188165, 2)
target mean: 0.19898210137996994
original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv (101371, 16)
weather: /kaggle/input/datasets/woodshole/playground-series-s6-e5-weather-conditions/F1_Weather_2022_2025.csv (14896, 11)


,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,Round_Number,Year,Track
0,0 days 00:01:03.204000,25.6,17.0,1010.2,False,32.3,346,0.5,1,2022,Sakhir
1,0 days 00:02:03.202000,25.7,17.0,1010.2,False,32.3,347,0.6,1,2022,Sakhir
2,0 days 00:03:03.205000,25.7,17.0,1010.0,False,32.2,359,0.4,1,2022,Sakhir
3,0 days 00:04:03.220000,25.7,17.0,1010.2,False,32.2,8,0.4,1,2022,Sakhir
4,0 days 00:05:03.218000,25.6,17.0,1010.0,False,32.1,16,0.5,1,2022,Sakhir


In [6]:
# ============================================================
# Weather aggregate
# ============================================================

def make_weather_agg(weather_df: pd.DataFrame) -> pd.DataFrame:
    w = weather_df.copy()

    required = [
        "Year", "Track",
        "AirTemp", "Humidity", "Pressure", "Rainfall",
        "TrackTemp", "WindSpeed"
    ]
    missing = [c for c in required if c not in w.columns]
    assert not missing, f"weather missing columns: {missing}"

    w["Track_norm"] = w["Track"].map(norm_text)

    # Some weather files may store Rainfall as bool, int, float, or string.
    w["Rainfall_num"] = pd.to_numeric(w["Rainfall"], errors="coerce")
    if w["Rainfall_num"].isna().all():
        w["Rainfall_num"] = w["Rainfall"].astype(str).str.lower().isin(["true", "1", "yes", "rain"]).astype(float)
    w["Rainfall_num"] = w["Rainfall_num"].fillna(0.0)

    for c in ["AirTemp", "Humidity", "Pressure", "TrackTemp", "WindSpeed"]:
        w[c] = pd.to_numeric(w[c], errors="coerce")

    w["TrackTemp_minus_AirTemp"] = w["TrackTemp"] - w["AirTemp"]

    agg = w.groupby(["Year", "Track_norm"]).agg(
        AirTemp_mean=("AirTemp", "mean"),
        AirTemp_min=("AirTemp", "min"),
        AirTemp_max=("AirTemp", "max"),
        AirTemp_std=("AirTemp", "std"),
        TrackTemp_mean=("TrackTemp", "mean"),
        TrackTemp_min=("TrackTemp", "min"),
        TrackTemp_max=("TrackTemp", "max"),
        TrackTemp_std=("TrackTemp", "std"),
        Humidity_mean=("Humidity", "mean"),
        Humidity_min=("Humidity", "min"),
        Humidity_max=("Humidity", "max"),
        Humidity_std=("Humidity", "std"),
        Pressure_mean=("Pressure", "mean"),
        Pressure_std=("Pressure", "std"),
        WindSpeed_mean=("WindSpeed", "mean"),
        WindSpeed_max=("WindSpeed", "max"),
        WindSpeed_std=("WindSpeed", "std"),
        Rainfall_mean=("Rainfall_num", "mean"),
        Rainfall_max=("Rainfall_num", "max"),
        Rainfall_sum=("Rainfall_num", "sum"),
        Rainfall_count=("Rainfall_num", "count"),
        TrackTemp_minus_AirTemp_mean=("TrackTemp_minus_AirTemp", "mean"),
        TrackTemp_minus_AirTemp_max=("TrackTemp_minus_AirTemp", "max"),
    ).reset_index()

    agg["AirTemp_range"] = agg["AirTemp_max"] - agg["AirTemp_min"]
    agg["TrackTemp_range"] = agg["TrackTemp_max"] - agg["TrackTemp_min"]
    agg["Rainfall_any"] = (agg["Rainfall_max"] > 0).astype(int)
    agg["Rainfall_rate"] = agg["Rainfall_sum"] / agg["Rainfall_count"].clip(lower=1)
    agg["WetRace"] = (agg["Rainfall_any"] > 0).astype(int)

    # Prefix all weather features.
    key_cols = ["Year", "Track_norm"]
    feat_cols = [c for c in agg.columns if c not in key_cols]
    agg = agg[key_cols + feat_cols]
    agg = agg.rename(columns={c: f"W_{c}" for c in feat_cols})

    return agg


weather_agg = make_weather_agg(weather)
print("weather_agg:", weather_agg.shape)
display(weather_agg.head())

weather_agg: (92, 30)


,Year,Track_norm,W_AirTemp_mean,W_AirTemp_min,W_AirTemp_max,W_AirTemp_std,W_TrackTemp_mean,W_TrackTemp_min,W_TrackTemp_max,W_TrackTemp_std,W_Humidity_mean,W_Humidity_min,W_Humidity_max,W_Humidity_std,W_Pressure_mean,W_Pressure_std,W_WindSpeed_mean,W_WindSpeed_max,W_WindSpeed_std,W_Rainfall_mean,W_Rainfall_max,W_Rainfall_sum,W_Rainfall_count,W_TrackTemp_minus_AirTemp_mean,W_TrackTemp_minus_AirTemp_max,W_AirTemp_range,W_TrackTemp_range,W_Rainfall_any,W_Rainfall_rate,W_WetRace
0,2022,austin,29.285294,28.3,30.7,0.715811,35.523529,33.4,38.1,1.262256,49.623529,45.0,54.0,2.446959,991.199412,0.947903,5.487059,10.1,1.327389,0.000000,False,0,170,6.238235,7.7,2.4,4.7,0,0.000000,0
1,2022,baku,25.435404,24.9,26.1,0.307289,47.149689,40.9,50.9,1.833444,59.490683,52.0,65.0,3.748530,1009.901242,0.486953,1.529193,2.9,0.539866,0.000000,False,0,161,21.714286,25.2,1.2,10.0,0,0.000000,0
2,2022,barcelona,36.565854,35.9,37.2,0.345956,48.957317,47.6,49.7,0.491531,7.109756,5.0,11.0,1.009235,996.015854,0.490891,3.488415,5.5,0.940917,0.000000,False,0,164,12.391463,13.1,1.3,2.1,0,0.000000,0
3,2022,budapest,18.559880,17.8,19.4,0.466474,26.720958,22.8,29.3,1.324188,69.317365,66.0,72.0,1.628399,985.581437,0.224849,4.034132,5.8,0.643295,0.275449,True,46,167,8.161078,10.1,1.6,6.5,1,0.275449,1
4,2022,imola,12.899367,11.1,14.2,0.787360,17.418987,15.6,19.1,0.992144,83.170886,75.0,92.0,4.909620,1001.038608,0.322159,1.482278,2.5,0.546036,0.000000,False,0,158,4.519620,7.4,3.1,3.5,0,0.000000,0


In [7]:
# ============================================================
# Race -> Track mapping
# ============================================================

def build_track_matcher(weather_df: pd.DataFrame):
    track_values = sorted(weather_df["Track"].dropna().astype(str).unique().tolist())
    track_norm_map = {norm_text(t): t for t in track_values}
    track_norms = sorted(track_norm_map.keys())

    # Aliases are intentionally broad. The final match is made against actual weather Track names.
    aliases = {
        "bahrain": ["bahrain", "sakhir"],
        "saudi arabian": ["jeddah", "saudi"],
        "australian": ["albert park", "melbourne", "australia"],
        "azerbaijan": ["baku", "azerbaijan"],
        "miami": ["miami"],
        "emilia romagna": ["imola", "emilia"],
        "monaco": ["monaco", "monte carlo"],
        "spanish": ["barcelona", "catalunya", "spain"],
        "canadian": ["gilles villeneuve", "montreal", "canada"],
        "austrian": ["red bull ring", "spielberg", "austria"],
        "british": ["silverstone", "britain", "great britain"],
        "hungarian": ["hungaroring", "hungary"],
        "belgian": ["spa", "francorchamps", "belgium"],
        "dutch": ["zandvoort", "netherlands", "dutch"],
        "italian": ["monza", "italy"],
        "singapore": ["marina bay", "singapore"],
        "japanese": ["suzuka", "japan"],
        "qatar": ["lusail", "qatar"],
        "united states": ["americas", "austin", "cota", "united states"],
        "mexico": ["hermanos rodriguez", "mexico"],
        "mexico city": ["hermanos rodriguez", "mexico"],
        "sao paulo": ["interlagos", "sao paulo", "brazil"],
        "brazilian": ["interlagos", "sao paulo", "brazil"],
        "las vegas": ["las vegas", "vegas"],
        "abu dhabi": ["yas marina", "abu dhabi"],
        "chinese": ["shanghai", "china"],
        "french": ["paul ricard", "france"],
        "portuguese": ["portimao", "portugal"],
        "turkish": ["istanbul", "turkey"],
        "styrian": ["red bull ring", "spielberg", "austria"],
        "eifel": ["nurburgring", "eifel"],
        "tuscan": ["mugello", "tuscany"],
    }

    def find_by_token(tokens):
        for token in tokens:
            tn = norm_text(token)
            for trn in track_norms:
                if tn and (tn in trn or trn in tn):
                    return trn
        return None

    def match_one(race):
        rn = norm_text(race)

        # Direct / substring.
        for trn in track_norms:
            if trn and (trn in rn or rn in trn):
                return trn, "direct"

        # Alias.
        for key, tokens in aliases.items():
            keyn = norm_text(key)
            if keyn and keyn in rn:
                hit = find_by_token(tokens)
                if hit is not None:
                    return hit, "alias"

        # Fuzzy fallback.
        best_trn = None
        best_score = -1
        for trn in track_norms:
            score = SequenceMatcher(None, rn, trn).ratio()
            if score > best_score:
                best_score = score
                best_trn = trn

        if best_score >= 0.42:
            return best_trn, f"fuzzy_{best_score:.3f}"

        return None, "unmatched"

    return match_one, track_norm_map


match_track, track_norm_map = build_track_matcher(weather)

race_values = sorted(pd.concat([train["Race"], test["Race"]], axis=0).dropna().astype(str).unique().tolist())
race_map_rows = []
for r in race_values:
    trn, method = match_track(r)
    race_map_rows.append({
        "Race": r,
        "Race_norm": norm_text(r),
        "Track_norm": trn,
        "Track_weather_name": track_norm_map.get(trn, None),
        "match_method": method,
    })

race_map = pd.DataFrame(race_map_rows)
race_map.to_csv(CFG.OUTDIR / "race_to_track_mapping.csv", index=False)

print("Race -> Track mapping")
display(race_map)

unmatched = race_map[race_map["Track_norm"].isna()]
if len(unmatched) > 0:
    print("WARNING: unmatched Race values")
    display(unmatched)

Race -> Track mapping


,Race,Race_norm,Track_norm,Track_weather_name,match_method
0,Abu Dhabi Grand Prix,abu dhabi,budapest,Budapest,fuzzy_0.471
1,Australian Grand Prix,australian,melbourne,Melbourne,alias
2,Austrian Grand Prix,austrian,spielberg,Spielberg,alias
3,Azerbaijan Grand Prix,azerbaijan,baku,Baku,alias
4,Bahrain Grand Prix,bahrain,sakhir,Sakhir,alias
5,Belgian Grand Prix,belgian,spa francorchamps,Spa-Francorchamps,alias
6,British Grand Prix,british,silverstone,Silverstone,alias
7,Canadian Grand Prix,canadian,shanghai,Shanghai,fuzzy_0.500
8,Chinese Grand Prix,chinese,shanghai,Shanghai,alias
9,Dutch Grand Prix,dutch,zandvoort,Zandvoort,alias


In [8]:
# ============================================================
# Base feature engineering
# ============================================================

def attach_weather(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.merge(race_map[["Race", "Track_norm"]], on="Race", how="left")
    out = out.merge(weather_agg, on=["Year", "Track_norm"], how="left")

    weather_cols = [c for c in out.columns if c.startswith("W_")]
    out["weather_missing"] = out[weather_cols].isna().all(axis=1).astype(int)

    # Missing weather values are mostly mapping/session coverage issues.
    # Fill by Year mean first, then global mean.
    for c in weather_cols:
        out[c] = out.groupby("Year")[c].transform(lambda s: s.fillna(s.mean()))
        out[c] = out[c].fillna(out[c].mean())
        out[c] = out[c].fillna(0.0)

    return out


def make_base_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if CFG.DROP_NORMALIZED_TYRELIFE:
        for c in list(out.columns):
            if "Normalized_TyreLife" in c:
                out = out.drop(columns=[c])

    if CFG.DROP_PITSTOP and "PitStop" in out.columns:
        out = out.drop(columns=["PitStop"])

    # Basic robust numerical transforms.
    if "LapTime_Delta" in out.columns:
        out["LapTime_Delta_abs"] = out["LapTime_Delta"].abs()

    if "TyreLife" in out.columns:
        out["TyreLife_sq"] = out["TyreLife"] ** 2
        out["TyreLife_sqrt"] = np.sqrt(np.clip(out["TyreLife"], 0, None))

    if "RaceProgress" in out.columns:
        rp = out["RaceProgress"].astype(float).clip(0, 1)
        out["RaceProgress_sq"] = rp ** 2
        out["RaceProgressBin_20"] = np.floor(rp * CFG.RACE_PROGRESS_BINS).astype(int).clip(0, CFG.RACE_PROGRESS_BINS - 1).astype(str)

    if "Race" in out.columns and "RaceProgressBin_20" in out.columns:
        out["Race_RaceProgressBin_20"] = out["Race"].astype(str) + "__rpbin" + out["RaceProgressBin_20"].astype(str)

    if "Race" in out.columns and "Compound" in out.columns and "RaceProgressBin_20" in out.columns:
        out["Race_Compound_RaceProgressBin_20"] = (
            out["Race"].astype(str) + "__" +
            out["Compound"].astype(str) + "__rpbin" +
            out["RaceProgressBin_20"].astype(str)
        )

    if "Race" in out.columns and "Compound" in out.columns:
        out["Race_Compound"] = out["Race"].astype(str) + "__" + out["Compound"].astype(str)

    if "Race" in out.columns and "Year" in out.columns:
        # Existing shared family uses Race_Year. Keep it.
        out["Race_Year"] = out["Race"].astype(str) + "__" + out["Year"].astype(str)

    # Weather interaction without target.
    if "W_TrackTemp_mean" in out.columns and "W_AirTemp_mean" in out.columns:
        out["W_TrackTemp_AirTemp_ratio"] = safe_div(out["W_TrackTemp_mean"], out["W_AirTemp_mean"])

    if "W_Rainfall_any" in out.columns and "Compound" in out.columns:
        out["Compound_x_WetRace"] = out["Compound"].astype(str) + "__wet" + out["W_Rainfall_any"].astype(int).astype(str)

    # Drop helper columns.
    drop_cols = [CFG.ID_COL, CFG.TARGET, "Track_norm"]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    return out

In [9]:
# ============================================================
# Fold-safe frequency/count and TE
# ============================================================

def add_count_freq_from_train(X_tr, X_va, X_te, cols):
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    for col in cols:
        if col not in X_tr.columns:
            continue

        vc = X_tr[col].astype(str).value_counts(dropna=False)
        total = len(X_tr)

        for X in [X_tr, X_va, X_te]:
            key = X[col].astype(str)
            X[f"{col}_count"] = key.map(vc).fillna(0).astype(np.float32)
            X[f"{col}_freq"] = (X[f"{col}_count"] / max(total, 1)).astype(np.float32)

    return X_tr, X_va, X_te


def make_target_encoding_oof(X_tr, y_tr, X_va, X_te, cols, n_splits=5, seed=42, smooth=20.0):
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    y_tr = np.asarray(y_tr).astype(float)
    global_mean = float(np.mean(y_tr))

    inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for col in cols:
        if col not in X_tr.columns:
            continue

        new_col = f"{col}_TE"
        X_tr[new_col] = np.nan

        values = X_tr[col].astype(str).values

        for in_tr_idx, in_va_idx in inner.split(X_tr, y_tr):
            tmp = pd.DataFrame({
                "key": values[in_tr_idx],
                "y": y_tr[in_tr_idx],
            })
            stat = tmp.groupby("key")["y"].agg(["sum", "count"])
            enc = (stat["sum"] + global_mean * smooth) / (stat["count"] + smooth)

            X_tr.iloc[in_va_idx, X_tr.columns.get_loc(new_col)] = (
                pd.Series(values[in_va_idx]).map(enc).fillna(global_mean).values
            )

        # Full train mapping for val/test.
        tmp_full = pd.DataFrame({"key": values, "y": y_tr})
        stat_full = tmp_full.groupby("key")["y"].agg(["sum", "count"])
        enc_full = (stat_full["sum"] + global_mean * smooth) / (stat_full["count"] + smooth)

        X_va[new_col] = X_va[col].astype(str).map(enc_full).fillna(global_mean).astype(np.float32)
        X_te[new_col] = X_te[col].astype(str).map(enc_full).fillna(global_mean).astype(np.float32)
        X_tr[new_col] = X_tr[new_col].fillna(global_mean).astype(np.float32)

    return X_tr, X_va, X_te


def prepare_lgb_dtypes(X_tr, X_va, X_te):
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    for col in X_tr.columns:
        # Convert bool to int.
        if X_tr[col].dtype == bool:
            X_tr[col] = X_tr[col].astype(np.int8)
            X_va[col] = X_va[col].astype(np.int8)
            X_te[col] = X_te[col].astype(np.int8)

        # Object columns as category for LightGBM.
        if str(X_tr[col].dtype) == "object":
            all_vals = pd.concat([X_tr[col], X_va[col], X_te[col]], axis=0).astype(str).fillna("__NA__")
            cats = pd.Categorical(all_vals).categories
            X_tr[col] = pd.Categorical(X_tr[col].astype(str).fillna("__NA__"), categories=cats)
            X_va[col] = pd.Categorical(X_va[col].astype(str).fillna("__NA__"), categories=cats)
            X_te[col] = pd.Categorical(X_te[col].astype(str).fillna("__NA__"), categories=cats)

    cat_cols = [c for c in X_tr.columns if str(X_tr[c].dtype) == "category"]

    # Numeric NaNs are OK in LightGBM, but inf is not.
    for X in [X_tr, X_va, X_te]:
        num_cols = [c for c in X.columns if c not in cat_cols]
        X[num_cols] = X[num_cols].replace([np.inf, -np.inf], np.nan)

    return X_tr, X_va, X_te, cat_cols

In [10]:
# ============================================================
# Precompute weather-attached frames
# ============================================================

train_w = attach_weather(train)
test_w = attach_weather(test)

orig_w = None
if orig is not None:
    # Keep only columns that are compatible with competition feature space plus target.
    # Never use Normalized_TyreLife even if original has it.
    if CFG.DROP_NORMALIZED_TYRELIFE:
        orig = orig.drop(columns=[c for c in orig.columns if "Normalized_TyreLife" in c], errors="ignore")

    # If original lacks id, create dummy id only for processing.
    if CFG.ID_COL not in orig.columns:
        orig[CFG.ID_COL] = -np.arange(len(orig)) - 1

    common_cols = [c for c in train.columns if c in orig.columns]
    if CFG.TARGET in orig.columns and len(common_cols) >= 5:
        orig_use = orig[common_cols].copy()
        orig_w = attach_weather(orig_use)
        print("orig_w:", orig_w.shape)
    else:
        print("original skipped: incompatible columns.")


# Diagnostics.
weather_cols = [c for c in train_w.columns if c.startswith("W_")]
print("weather feature count:", len(weather_cols))
print("train weather_missing mean:", train_w["weather_missing"].mean())
print("test weather_missing mean :", test_w["weather_missing"].mean())
if orig_w is not None:
    print("orig weather_missing mean :", orig_w["weather_missing"].mean())

orig_w: (101371, 46)
weather feature count: 28
train weather_missing mean: 0.09400191282962153
test weather_missing mean : 0.09381659713549279
orig weather_missing mean : 0.10647029229266752


In [11]:
# ============================================================
# CV training
# ============================================================

y = train[CFG.TARGET].astype(int).values
test_pred = np.zeros(len(test), dtype=np.float32)
oof = np.zeros(len(train), dtype=np.float32)
fold_scores = []
best_iterations = []
feature_importance_rows = []

skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

orig_splits = None
if orig_w is not None and CFG.USE_ORIGINAL_APPEND:
    y_orig = orig_w[CFG.TARGET].astype(int).values
    orig_splits = list(StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED).split(orig_w, y_orig))
else:
    y_orig = None

t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_w, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 80)

    fold_train = train_w.iloc[tr_idx].copy()
    fold_valid = train_w.iloc[va_idx].copy()

    if orig_w is not None and CFG.USE_ORIGINAL_APPEND:
        orig_tr_idx, _ = orig_splits[fold - 1]
        fold_train = pd.concat([fold_train, orig_w.iloc[orig_tr_idx].copy()], axis=0, ignore_index=True)
        print("fold_train with original:", fold_train.shape)

    y_tr = fold_train[CFG.TARGET].astype(int).values
    y_va = fold_valid[CFG.TARGET].astype(int).values

    X_tr = make_base_features(fold_train)
    X_va = make_base_features(fold_valid)
    X_te = make_base_features(test_w)

    count_freq_cols = [c for c in CFG.TE_COLS if c in X_tr.columns]
    X_tr, X_va, X_te = add_count_freq_from_train(X_tr, X_va, X_te, count_freq_cols)

    X_tr, X_va, X_te = make_target_encoding_oof(
        X_tr, y_tr, X_va, X_te,
        cols=[c for c in CFG.TE_COLS if c in X_tr.columns],
        n_splits=CFG.INNER_SPLITS,
        seed=CFG.SEED + fold,
        smooth=CFG.TE_SMOOTH,
    )

    X_tr, X_va, X_te, cat_cols = prepare_lgb_dtypes(X_tr, X_va, X_te)

    if fold == 1:
        print("feature count:", X_tr.shape[1])
        print("categorical count:", len(cat_cols))
        print("categorical cols:", cat_cols[:50])
        pd.Series(X_tr.columns).to_csv(CFG.OUTDIR / "feature_columns.csv", index=False, header=False)

    model = lgb.LGBMClassifier(**CFG.LGB_PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        categorical_feature=cat_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=300, verbose=True),
            lgb.log_evaluation(period=200),
        ],
    )

    best_it = int(model.best_iteration_ or CFG.LGB_PARAMS["n_estimators"])
    best_iterations.append(best_it)

    pred_va = model.predict_proba(X_va, num_iteration=best_it)[:, 1]
    pred_te = model.predict_proba(X_te, num_iteration=best_it)[:, 1]

    oof[va_idx] = pred_va.astype(np.float32)
    test_pred += pred_te.astype(np.float32) / CFG.N_SPLITS

    fold_auc = auc(y_va, pred_va)
    fold_scores.append(float(fold_auc))

    print(f"Fold {fold} AUC: {fold_auc:.9f} | best_it={best_it} | elapsed={(time.time()-t0)/60:.1f} min")

    fi = pd.DataFrame({
        "feature": X_tr.columns,
        "importance_gain": model.booster_.feature_importance(importance_type="gain"),
        "importance_split": model.booster_.feature_importance(importance_type="split"),
        "fold": fold,
    })
    feature_importance_rows.append(fi)

    np.save(CFG.OUTDIR / f"oof_{CFG.TAG}_fold{fold:02d}.npy", oof)
    np.save(CFG.OUTDIR / f"pred_{CFG.TAG}_fold{fold:02d}.npy", pred_te.astype(np.float32))
    np.save(CFG.OUTDIR / f"train_idx_{CFG.TAG}_fold{fold:02d}.npy", np.asarray(tr_idx))
    np.save(CFG.OUTDIR / f"val_idx_{CFG.TAG}_fold{fold:02d}.npy", np.asarray(va_idx))

    del X_tr, X_va, X_te, fold_train, fold_valid, model, pred_va, pred_te
    gc.collect()


Fold 1/5
fold_train with original: (432408, 46)
feature count: 62
categorical count: 10
categorical cols: ['Driver', 'Compound', 'Race', 'W_Rainfall_max', 'RaceProgressBin_20', 'Race_RaceProgressBin_20', 'Race_Compound_RaceProgressBin_20', 'Race_Compound', 'Race_Year', 'Compound_x_WetRace']
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.942332
[400]	valid_0's auc: 0.944114
[600]	valid_0's auc: 0.944605
[800]	valid_0's auc: 0.944726
[1000]	valid_0's auc: 0.944463
Early stopping, best iteration is:
[801]	valid_0's auc: 0.944737
Fold 1 AUC: 0.944736834 | best_it=801 | elapsed=1.7 min

Fold 2/5
fold_train with original: (432409, 46)
Training until validation scores don't improve for 300 rounds
[200]	valid_0's auc: 0.940392
[400]	valid_0's auc: 0.94183
[600]	valid_0's auc: 0.94235
[800]	valid_0's auc: 0.94247
[1000]	valid_0's auc: 0.942126
Early stopping, best iteration is:
[767]	valid_0's auc: 0.942509
Fold 2 AUC: 0.942509274 | best_it=767 | elapsed=3

In [12]:
# ============================================================
# Results / save artifacts
# ============================================================

overall_auc = auc(y, oof)
mean_auc = float(np.mean(fold_scores))
std_auc = float(np.std(fold_scores))

print("\n" + "=" * 80)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 80)
print(f"Overall CV AUC: {overall_auc:.9f}")
print(f"Mean Fold AUC : {mean_auc:.9f} +/- {std_auc:.9f}")
print(f"Best it mean  : {np.mean(best_iterations):.1f}")
print("Fold scores   :", fold_scores)

np.save(CFG.OUTDIR / f"oof_exp_20260513_024_lgb_progress_window_weather_agg_min.npy", oof)
np.save(CFG.OUTDIR / f"pred_exp_20260513_024_lgb_progress_window_weather_agg_min.npy", test_pred)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]
sub_out = sub.copy()
sub_out[target_col] = np.clip(test_pred, 1e-6, 1 - 1e-6)
sub_path = CFG.OUTDIR / "submission_exp_20260513_024_lgb_progress_window_weather_agg_min.csv"
sub_out.to_csv(sub_path, index=False)

fi_all = pd.concat(feature_importance_rows, axis=0, ignore_index=True)
fi_summary = (
    fi_all.groupby("feature")[["importance_gain", "importance_split"]]
    .mean()
    .sort_values("importance_gain", ascending=False)
    .reset_index()
)
fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "tag": CFG.TAG,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-13",
    },
    "base": "exp_20260511_020_lgb_progress_window_te_min",
    "objective": "Add Race-Year weather aggregate features to 020 LGB progress-window TE.",
    "do_not_add": [
        "Normalized_TyreLife",
        "Normalized_TyreLife proxy",
        "Weather x target TE",
        "Time x RaceProgress interpolation",
        "Race_Year_RaceProgressBin",
        "pseudo label",
    ],
    "cv": {
        "overall_auc": float(overall_auc),
        "mean_fold_auc": mean_auc,
        "std_fold_auc": std_auc,
        "fold_scores": fold_scores,
        "best_iterations": [int(x) for x in best_iterations],
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_path": None if original_path is None else str(original_path),
        "weather_path": str(weather_path),
        "weather_shape": list(weather.shape),
        "weather_agg_shape": list(weather_agg.shape),
        "train_weather_missing_mean": float(train_w["weather_missing"].mean()),
        "test_weather_missing_mean": float(test_w["weather_missing"].mean()),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof": str(CFG.OUTDIR / "oof_exp_20260513_024_lgb_progress_window_weather_agg_min.npy"),
        "pred": str(CFG.OUTDIR / "pred_exp_20260513_024_lgb_progress_window_weather_agg_min.npy"),
        "submission": str(sub_path),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "race_to_track_mapping": str(CFG.OUTDIR / "race_to_track_mapping.csv"),
    },
    "notes": [
        "Weather is joined only at Year + Track level after Race->Track mapping.",
        "No exact Time/RaceProgress interpolation is used.",
        "No Weather x target TE is used.",
        "PitStop is dropped to preserve the 020-family policy.",
        "Original rows are appended to fold train side if compatible original data is found.",
    ],
}

with open(CFG.OUTDIR / "result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("saved:", CFG.OUTDIR)
print("submission:", sub_path)
display(fi_summary.head(50))


MODEL PERFORMANCE SUMMARY
Overall CV AUC: 0.943121690
Mean Fold AUC : 0.943125375 +/- 0.000921593
Best it mean  : 784.2
Fold scores   : [0.9447368344150838, 0.9425092740982753, 0.9433797537757099, 0.942039717076756, 0.9429612959438365]
saved: /kaggle/working/exp_20260513_024_lgb_progress_window_weather_agg_min
submission: /kaggle/working/exp_20260513_024_lgb_progress_window_weather_agg_min/submission_exp_20260513_024_lgb_progress_window_weather_agg_min.csv


,feature,importance_gain,importance_split
0,Race_Year,1.453462e+06,4131.0
1,Race_Compound_RaceProgressBin_20_TE,1.302017e+06,282.0
2,Race_Compound_RaceProgressBin_20,7.810821e+05,14305.0
3,Driver,7.556879e+05,15048.8
4,TyreLife,5.070460e+05,1645.4
5,Stint,4.763015e+05,626.4
6,Race_RaceProgressBin_20,4.724748e+05,8310.4
7,LapTime_Delta_abs,2.031161e+05,499.4
8,Race_Compound,1.668863e+05,1072.6
9,Year,1.224058e+05,97.8


In [13]:
# ============================================================
# memo.yml draft
# ============================================================

memo_yml = f"""experiment:
  id: exp_20260513_024_lgb_progress_window_weather_agg_min
  title: LGB progress-window TE + Race-Year weather aggregate
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    020 LGB progress-window TE をbaseに、Woods Hole FastF1 weather dataset由来の
    Race-Year単位 weather aggregate を追加し、weatherが既存stackに対して
    低相関補助素材になるか確認する。
  base_experiment: exp_20260511_020_lgb_progress_window_te_min
  intended_role: aux_material

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  weather_dataset: /kaggle/input/playground-series-s6-e5-weather-conditions/F1_Weather_2022_2025.csv
  original_append: {str(CFG.USE_ORIGINAL_APPEND).lower()}
  target: PitNextLap

features:
  inherited_from_020:
    - RaceProgressBin_20
    - Race_RaceProgressBin_20
    - Race_Compound_RaceProgressBin_20
    - fold_safe_count_frequency
    - fold_safe_TE
  added_weather_aggregate:
    - AirTemp_mean_min_max_std
    - TrackTemp_mean_min_max_std
    - Humidity_mean_min_max_std
    - Pressure_mean_std
    - WindSpeed_mean_max_std
    - Rainfall_mean_max_sum_any_rate
    - TrackTemp_minus_AirTemp_mean_max
    - TrackTemp_range
    - AirTemp_range
    - WetRace
  join_policy: >
    competition側の Year + Race を Race->Track mapping で Year + Track に変換し、
    weather側の Year + Track aggregate をmergeする。
  not_used:
    - Normalized_TyreLife
    - Normalized_TyreLife_proxy
    - Weather_x_target_TE
    - Time_RaceProgress_interpolation
    - Race_Year_RaceProgressBin
    - pseudo_label

model:
  family: LightGBM
  seed: 42
  cv:
    scheme: StratifiedKFold
    n_splits: 5
    shuffle: true
    random_state: 42
  params:
    objective: binary
    metric: auc
    learning_rate: 0.02
    n_estimators: 8000
    num_leaves: 64
    min_child_samples: 80
    subsample: 0.85
    colsample_bytree: 0.80
    reg_alpha: 0.15
    reg_lambda: 2.0

result:
  cv_auc: null
  public_lb: null
  fold_scores: null
  compared_to_020:
    cv_020: 0.9510267
    public_lb_020: 0.95075
    delta_cv: null
    delta_lb: null

artifacts:
  notebook: exp_20260513_024_lgb_progress_window_weather_agg_min.ipynb
  oof: oof_exp_20260513_024_lgb_progress_window_weather_agg_min.npy
  pred: pred_exp_20260513_024_lgb_progress_window_weather_agg_min.npy
  submission: submission_exp_20260513_024_lgb_progress_window_weather_agg_min.csv
  feature_importance: feature_importance.csv
  race_to_track_mapping: race_to_track_mapping.csv

blend_policy:
  benchmark_code: code_010_add_slsqp.txt
  compare_methods:
    - nm_softmax_raw
    - hc_nonnegative_raw
    - logreg_meta_all
    - slsqp_signed_raw_neg005
  add_or_replace: >
    020置換候補として評価する。
    単体CVが020を明確に上回る、またはCV同等でも既存stackとの相関が下がる場合のみ
    blend benchmarkに投入する。

judgement:
  status: pending
  summary: >
    CV/LB/相関確認後に keep / hold / drop を判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("memo draft saved:", CFG.OUTDIR / "memo_draft.yml")

memo draft saved: /kaggle/working/exp_20260513_024_lgb_progress_window_weather_agg_min/memo_draft.yml
